# 00 · Colab setup
Rebuilds a working SecureMCP session on a free Colab T4. Use **Runtime → Run all**.
Steps: GPU check → mount Drive → clone repo → install deps → cache embedding model → smoke test.

**Private repo?** Add a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar,
a fine-grained token with read access to the repo) and enable *Notebook access*.

In [ ]:
# 1. GPU check
import subprocess, shutil
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
else:
    print("No GPU. Runtime → Change runtime type → T4 GPU, then Run all again.")

In [ ]:
# 2. Mount Google Drive and create the persistent folders
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/securemcp"
for sub in ["data", "models", "llm_cache", "results", "hf_cache"]:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

# Keep downloaded Hugging Face models on Drive so they survive disconnects
os.environ["HF_HOME"] = f"{DRIVE_ROOT}/hf_cache"
print("Drive ready:", DRIVE_ROOT)

In [ ]:
# 3. Clone (or update) the repo
import os, subprocess
REPO_OWNER = "RajyalakshmiKillamsetty2023"
REPO_NAME = "securemcp"
REPO_DIR = f"/content/{REPO_NAME}"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

url = (f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git" if token
       else f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git")

if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

# Do not leave the token in the saved remote URL
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin",
                f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-3"],
                     capture_output=True, text=True).stdout)

In [ ]:
# 4. Install dependencies
# Colab already ships torch with CUDA, so we install only what the project needs.
# (No editable install: Colab's Python version may be newer than the laptop's 3.11 pin.)
%pip install -q sentence-transformers scikit-learn pandas datasets \
    pydantic pydantic-settings pyyaml httpx

import sys
sys.path.insert(0, f"{REPO_DIR}/backend")

In [ ]:
# 5. Download and cache the embedding model (~90 MB, saved to Drive)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Loaded on:", model.device)

In [ ]:
# 6. Smoke test
from sentence_transformers import util
from app.config import settings

q = "What is the weather in Hyderabad tomorrow?"
good = "Returns the weather forecast for a city."
bad = "Converts a PDF file to Word format."
emb = model.encode([q, good, bad], convert_to_tensor=True)
s_good = float(util.cos_sim(emb[0], emb[1]))
s_bad = float(util.cos_sim(emb[0], emb[2]))
print(f"relevant={s_good:.3f}  irrelevant={s_bad:.3f}")
assert s_good > s_bad, "Embedding sanity check failed"
print("Config loaded, mongo_db =", settings.mongo_db)
print("\nSetup OK. Drive root:", DRIVE_ROOT)